In [0]:
bearscash_collection_staging=dbutils.widgets.get("bearscash_collection_staging")
bearscash_collection=dbutils.widgets.get("bearscash_collection")
cash_collection=dbutils.widgets.get("cash_collection")
cash_stg=dbutils.widgets.get("cash_stg")
date_path=dbutils.widgets.get("date_path")
cubeserviceofficetxnweekendingdate=dbutils.widgets.get("cubeserviceofficetxnweekendingdate")
payerdimension=dbutils.widgets.get("payerdimension")
office=dbutils.widgets.get("office")
cubeserviceofficetxnsourcesystem=dbutils.widgets.get("cubeserviceofficetxnsourcesystem")
client=dbutils.widgets.get("client")
paymentstype=dbutils.widgets.get("paymenttype")
paymentsdetails=dbutils.widgets.get("paymentsdetails")
mart_cash=dbutils.widgets.get("mart_cash")

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {cash_collection}").collect()[0]['cnt']

if count == 0:
    print(f"Full load for {cash_collection} executed")

    spark.sql(f"""
    INSERT INTO {cash_collection} (
        reporting_week_ending_date_key,
        posted_date_key,
        deposit_date_key,
        source_system_key,
        office_key,
        payor_key,
        client_key,
        payment_type_key,
        payment_detail_key,
        invoice_number,
        cash_collected
    )
    SELECT
        CAST(Reporting_Week_Ending_Date_Key AS INT),
        CAST(Posted_Date_Key AS INT),
        CAST(Deposit_Date_Key AS INT),
        CAST(Source_System_Key AS INT),
        CAST(Office_Key AS INT),
        CAST(Payor_Key AS INT),
        CAST(Client_Key AS INT),
        CAST(Payment_Type_Key AS INT),
        CAST(Payment_Detail_Key AS INT),
        CAST(Invoice_Number AS STRING),
        CAST(Cash_Collected AS DOUBLE)
    FROM {mart_cash}
    WHERE Source_System_Key IN (0, 19);
    """)


In [0]:
spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW Bearscash AS
SELECT 
  CASE 
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 1 THEN TO_DATE(PostedDate, 'MM-dd-yy')
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 2 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), -1)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 3 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), -2)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 4 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), -3)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 5 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), 3)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 6 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), 2)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 7 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), 1)
  END AS ReportingWeekendingDate,
  'BEARS' AS SourceSystem,
  PayorTypeCode,
  OfficeNumber,
  SUM(AppliedOnMR) * -1.00 AS CashCollected,
  PayorID,
  BillToName,
  ClientNumber,
  PostedDate,
  DepositDate,
  BatchID,
  CheckID,
  InvoiceNumber,
  Type,
  BatchNumber,
  Bank,
  Product
FROM {cash_stg} C
GROUP BY 
  CASE DAYOFWEEK(PostedDate)
    WHEN 1 THEN PostedDate
    WHEN 2 THEN DATE_ADD(PostedDate, -1)
    WHEN 3 THEN DATE_ADD(PostedDate, -2)
    WHEN 4 THEN DATE_ADD(PostedDate, -3)
    WHEN 5 THEN DATE_ADD(PostedDate, 3)
    WHEN 6 THEN DATE_ADD(PostedDate, 2)
    WHEN 7 THEN DATE_ADD(PostedDate, 1)
  END,
  PayorTypeCode,
  OfficeNumber,
  PayorID,
  BillToName,
  ClientNumber,
  PostedDate,
  DepositDate,
  BatchID,
  CheckID,
  InvoiceNumber,
  Type,
  BatchNumber,
  Bank,
  Product;
""" 
)

spark.sql(
  f""" 
INSERT INTO {paymentstype}(paymenttypedescription, sourcesystem, transactiontype)
SELECT DISTINCT 
  Product AS paymenttypedescription,
  'BEARS' AS sourcesystem,
  NULL AS transactiontype
FROM Bearscash
WHERE Product NOT IN (
  SELECT paymenttypedescription 
  FROM {paymentstype} 
  WHERE sourcesystem = 'BEARS'
);
    """
)

spark.sql(
    f"""
INSERT INTO {paymentsdetails}(batchid, type, bank, batchnumber, checkid, agencyid, depositid, sourcesystem)
SELECT DISTINCT 
  ca.BatchID AS batchid,
  ca.Type AS type,
  ca.Bank AS bank,
  ca.BatchNumber AS batchnumber,
  ca.CheckID AS checkid,
  NULL AS agencyid,
  NULL AS depositid,
  'BEARS' AS sourcesystem
FROM Bearscash ca
WHERE NOT EXISTS (
  SELECT 1 
  FROM {paymentsdetails} pl
  WHERE COALESCE(pl.batchid, '') = COALESCE(ca.BatchID, '')
    AND COALESCE(pl.checkid, '') = COALESCE(ca.CheckID, '')
    AND COALESCE(pl.type, '') = COALESCE(ca.Type, '')
    AND COALESCE(pl.bank, '') = COALESCE(ca.Bank, '')
    AND COALESCE(pl.batchnumber, '') = COALESCE(ca.BatchNumber, '')
    AND pl.sourcesystem = 'BEARS'
);
    """
)


spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW Keyslookup AS
SELECT 
  ca.ReportingWeekendingDate AS reportingweekendingdate,
  W.WeekendingDateKey AS reportingweekendingdatekey,
  SS.SourceSystemKey AS sourcesystemkey,
  ca.OfficeNumber AS officenumber,
  Oc.OfficeKey AS officekey,
  ca.PayorID AS payorid,
  CASE WHEN P.PayerKey IS NULL THEN -1 ELSE P.PayerKey END AS payorkey,
  ca.ClientNumber AS clientnumber,
  CASE WHEN Cl.ClientKey IS NULL THEN -1 ELSE Cl.ClientKey END AS clientkey,
  ca.PostedDate AS posteddate,
  CASE WHEN pd.DateKey IS NULL THEN -1 ELSE pd.DateKey END AS posteddatetkey,
  ca.DepositDate AS depositdate,
  CASE WHEN dd.DateKey IS NULL THEN -1 ELSE dd.DateKey END AS depositdatekey,
  ca.InvoiceNumber AS invoicenumber,
  CAST(ca.CashCollected AS DECIMAL(15,2)) AS cashcollected,
  ca.Product AS product,
  CASE WHEN PT.PaymentTypeKey IS NULL THEN -1 ELSE PT.PaymentTypeKey END AS paymenttypekey,
  ca.BatchID AS batchid,
  ca.Bank AS bank,
  ca.BatchNumber AS batchnumber,
  ca.Type AS type,
  ca.CheckID AS checkid,
  CAST(PL.PaymentDetailKey AS BIGINT) AS paymentdetailkey
FROM Bearscash ca
LEFT JOIN {date_path} pd 
  ON pd.CalendarDate = ca.PostedDate
LEFT JOIN {date_path} dd 
  ON dd.CalendarDate = ca.DepositDate
LEFT JOIN {cubeserviceofficetxnweekendingdate} W 
  ON W.WeekEndingDate = ca.ReportingWeekendingDate
LEFT JOIN {payerdimension} P 
  ON P.PayerID = ca.PayorID
LEFT JOIN {office} Oc 
  ON Oc.OfficeNumber = ca.OfficeNumber
LEFT JOIN {cubeserviceofficetxnsourcesystem} SS 
  ON SS.SourceSystemName = ca.SourceSystem
LEFT JOIN {client} CL
  ON CL.SourceSystemId = ca.ClientNumber 
  AND CL.OfficeNumber = ca.OfficeNumber 
  AND CL.SourceSystem = 'BEARS'
LEFT JOIN {paymentstype} PT
  ON COALESCE(PT.paymenttypedescription, '') = COALESCE(ca.Product, '')
LEFT JOIN {paymentsdetails} PL
  ON COALESCE(PL.batchid, '') = COALESCE(ca.BatchID, '')
  AND COALESCE(PL.checkid, '') = COALESCE(ca.CheckID, '')
  AND COALESCE(PL.type, '') = COALESCE(ca.Type, '')
  AND COALESCE(PL.bank, '') = COALESCE(ca.Bank, '')
  AND COALESCE(PL.batchnumber, '') = COALESCE(ca.BatchNumber, '')
  AND PL.sourcesystem = 'BEARS'
WHERE PT.transactiontype = 'Payment';
"""
)


In [0]:
spark.sql(
    f"""
INSERT INTO {bearscash_collection} (
    reportingweekendingdate,
    reporting_week_ending_date_key,
    source_system_key,
    serviceofficenumber,
    office_key,
    payorid,
    payor_key,
    clientnumber,
    client_key,
    posteddate,
    posted_date_key,
    depositdate,
    deposit_date_key,
    invoicenumber,
    cashcollected,
    product,
    payment_type_key,
    batchid,
    bank,
    batchnumber,
    type,
    checkid,
    paymentdetailkey
)
SELECT 
    reportingweekendingdate,
    reportingweekendingdatekey      AS reporting_week_ending_date_key,
    sourcesystemkey                 AS source_system_key,
    officenumber                    AS serviceofficenumber,
    officekey                       AS office_key,
    payorid,
    payorkey                        AS payor_key,
    clientnumber,
    clientkey                       AS client_key,
    posteddate,
    posteddatetkey                  AS posted_date_key,
    depositdate,
    depositdatekey                  AS deposit_date_key,
    invoicenumber,
    cashcollected,
    product,
    paymenttypekey                  AS payment_type_key,
    batchid,
    bank,
    batchnumber,
    type,
    checkid,
    paymentdetailkey
FROM Keyslookup
    """
)


In [0]:
spark.sql(
    f"""
TRUNCATE TABLE {bearscash_collection_staging};
 """
)

spark.sql(
    f"""
INSERT INTO {bearscash_collection_staging} (
    reportingweekendingdate,
    reporting_week_ending_date_key,
    source_system_key,
    serviceofficenumber,
    office_key,
    payorid,
    payor_key,
    clientnumber,
    client_key,
    posteddate,
    posted_date_key,
    depositdate,
    deposit_date_key,
    invoicenumber,
    cashcollected,
    product,
    payment_type_key,
    batchid,
    bank,
    batchnumber,
    type,
    checkid,
    paymentdetailkey
)
SELECT 
    reportingweekendingdate,
    reportingweekendingdatekey      AS reporting_week_ending_date_key,
    sourcesystemkey                 AS source_system_key,
    officenumber                    AS serviceofficenumber,
    officekey                       AS office_key,
    payorid,
    payorkey                        AS payor_key,
    clientnumber,
    clientkey                       AS client_key,
    posteddate,
    posteddatetkey                  AS posted_date_key,
    depositdate,
    depositdatekey                  AS deposit_date_key,
    invoicenumber,
    cashcollected,
    product,
    paymenttypekey                  AS payment_type_key,
    batchid,
    bank,
    batchnumber,
    type,
    checkid,
    paymentdetailkey
FROM Keyslookup
    """
)

In [0]:
spark.sql(
    f"""
INSERT INTO {cash_collection} (
    reporting_week_ending_date_key,
    posted_date_key,
    deposit_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    payment_type_key,
    payment_detail_key,
    invoice_number,
    cash_collected 
)
SELECT 
    CAST(reporting_week_ending_date_key AS INT),
    CAST(posted_date_key AS INT),
    CAST(deposit_date_key AS INT),
    CAST(source_system_key AS TINYINT),
    CAST(office_key AS INT),
    CAST(payor_key AS INT),
    CAST(client_key AS INT),
    CAST(payment_type_key AS INT),
    CAST(paymentdetailkey AS INT) AS payment_detail_key,
    CAST(invoicenumber AS STRING) AS invoice_number,
    CAST(cashcollected AS DOUBLE) AS cash_collected 
FROM {bearscash_collection_staging}
"""
)